# 探針實驗:在 Colab 跑(長庚 MercuryAPI)

流程:**設定 → 連線測試 → 小量試跑 → 完整實驗 → 分析**。日誌存在 Google Drive,Colab 斷線後重新執行同一格就會**自動續跑**。

事前準備:
1. 到 MercuryAPI 領取金鑰(格式 `sk-mercury-…`)。左側 🔑「密鑰」新增 `MERCURY_API_KEY`,並開啟「筆記本存取權」。**不要把金鑰貼進程式格。**
2. **本地模型(gpt-oss-120b、gemma-4 等)免費,不計入額度,也不受每分鐘次數限制**,每人同時最多 8 個請求。預設用這些。
3. 付費的 gpt 系列:每分鐘 30 次、同時 2 個、每週 $10。本筆記本預設不用,需要時在第 3 格啟用。
4. 注意:透過金鑰送出的請求內容會匿名保存供研究,不要放密碼或機密。

## 1. 安裝與取得程式

In [ ]:
import os, subprocess, sys
REPO = "Ting-liu0103/coop-llm"      # 改成你的 repo
tok = ""
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass
url = f"https://{tok + '@' if tok else ''}github.com/{REPO}.git"
if not os.path.isdir("/content/coop-llm"):
    subprocess.run(["git", "clone", "--depth", "1", url, "/content/coop-llm"], check=True)
else:
    subprocess.run(["git", "-C", "/content/coop-llm", "pull", "--ff-only"], check=True)
os.chdir("/content/coop-llm")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[openai,anthropic,plots,dev]"], check=True)
# 讓目前這個 Python 程序也找得到 coop_llm(新安裝的套件在已啟動的 kernel 裡不一定立刻可 import)
import importlib
for pth in ("/content/coop-llm/src",):
    if pth not in sys.path:
        sys.path.insert(0, pth)
importlib.invalidate_caches()
import coop_llm
print("OK", coop_llm.__file__)

## 2. 自我檢查(78 項測試,不花錢)

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(r.stdout[-400:])
assert r.returncode == 0, "測試沒全過,先不要往下跑"

## 3. 設定模型
預設用兩個**免費的開放權重模型**(不同家族):`gpt-oss-120b`、`gemma-4-26b-a4b-it`。可換成 `mistral-small-4-119b-2603`、`nemotron-3-super-120b-a12b`、`nemotron-3.5-bolt-74b-a9b`。
- 原本設計是「1 個商用小模型 + 1 個開源模型」。若想保留商用模型,把下面註解的 `gpt-6-luna` 打開(會花額度,先看試跑費用)。
- 這些模型可能先「思考」再回答,思考也佔 `max_tokens`。試跑時若回覆常為空或解析失敗,把 `max_tokens` 調大。

In [ ]:
import yaml
from google.colab import userdata

BASE = "https://air.cgu.edu.tw/mercuryapi/v1"
FREE = dict(provider="openai_compat", base_url=BASE, api_key_env="MERCURY_API_KEY",
            max_concurrent=8, max_tokens=1200)           # 本地模型:免費、不限每分鐘次數、同時最多 8 個
PAID = dict(provider="openai_compat", base_url=BASE, api_key_env="MERCURY_API_KEY",
            rpm=27, max_concurrent=2, max_tokens=1200)   # 付費模型:30 RPM、同時 2 個
MODELS = [
    {"name": "oss_120b", "model": "gpt-oss-120b", **FREE},
    {"name": "gemma_26b", "model": "gemma-4-26b-a4b-it", **FREE},
    # {"name": "api_small", "model": "gpt-6-luna", **PAID},   # 要加商用模型就取消註解(會花額度)
]
os.environ["MERCURY_API_KEY"] = userdata.get("MERCURY_API_KEY")
yaml.safe_dump({"models": MODELS}, open("configs/models_run.yaml", "w"), allow_unicode=True)
print([m["name"] for m in MODELS])

### 連線測試
對每個模型送一個最小請求。有些模型不接受自訂溫度(0.7);若被拒絕,這格會自動改成「不傳溫度」並更新設定。

In [ ]:
import asyncio, yaml
from coop_llm.llm import make_client

async def smoke(m):
    c = make_client(m)
    for temp in (0.7, None):
        try:
            r = await c.complete("你是測試員。", [{"role": "user", "content": '只回傳 JSON: {"ok": true}'}],
                                 temperature=temp, max_tokens=m.get("max_tokens", 1200))
            return temp, r.text[:80], r.tokens_in, r.tokens_out
        except Exception as e:
            print(f"  {m['name']} temperature={temp} 失敗: {type(e).__name__}: {str(e)[:160]}")
    return "ALL_FAILED", "", 0, 0

for m in MODELS:
    temp, text, tin, tout = await smoke(m)
    print(m["name"], "| temperature =", temp, "| 回覆:", repr(text), "| tokens", tin, tout)
    assert temp != "ALL_FAILED", f"{m['name']} 連不上,檢查金鑰、型號、網址"
    if temp is None:
        m["temperature"] = None   # 這個模型不傳溫度
yaml.safe_dump({"models": MODELS}, open("configs/models_run.yaml", "w"), allow_unicode=True)

## 4. 日誌存到 Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
LOGS = "/content/drive/MyDrive/coop-llm-logs/probe"
os.makedirs(LOGS, exist_ok=True)
import glob
CONFIGS = sorted(glob.glob("configs/probe/*.yaml"))
print(CONFIGS)

## 5. 試跑
**5a 迷你測試(只跑基準條件 P0,每格 1 場)**:確認格式、解析都正常。
**5b 完整試跑(全部 6 個條件,每格 2 場)**:5a 沒問題再跑。

若有啟用付費模型:試跑前先到「我的用量」記下「已用 $」,跑完再看一次,差額 ÷ 場次 = 每場成本。

In [ ]:
def run_all(reps, out, start_rep=0, configs=None):
    # 模型一個一個跑:同時請求數是整個帳號共用的上限,兩個模型同時跑會互相搶
    for m in MODELS:
        for c in (configs or CONFIGS):
            r = subprocess.run([sys.executable, "-m", "coop_llm.runner", "--condition", c, "--models",
                                "configs/models_run.yaml", "--only-model", m["name"], "--reps", str(reps),
                                "--start-rep", str(start_rep), "--out", out, "--concurrency", str(m.get("max_concurrent", 2))],
                               capture_output=True, text=True)
            tail = r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-300:]
            print(m["name"], c.split("/")[-1], "→", tail)

PILOT = "/content/drive/MyDrive/coop-llm-logs/pilot"
# 5a 迷你測試:只跑 P0、每格 1 場
run_all(1, PILOT, configs=[c for c in CONFIGS if "p0_" in c])

In [ ]:
# 5b 完整試跑(5a 沒問題再執行這格):全部條件、每格 2 場。已完成的場次會自動跳過。
run_all(2, PILOT)

In [ ]:
import pandas as pd
from coop_llm.analysis import load_logs, parse_failure_report
rounds, chats, summary = load_logs(PILOT)
print("完成場次:", summary["game_id"].nunique(), "(5a 預期 24 場;5b 預期 276 場)")
print(parse_failure_report(rounds).to_string())
print("輸入 token 合計:", int(rounds["tokens_in"].sum()), " 輸出 token 合計:", int(rounds["tokens_out"].sum()))
print("輸出 token 接近 max_tokens 的比例:", round((rounds["tokens_out"] >= 1100).mean(), 3), "← 偏高代表被截斷")
print("看一段想法:", rounds["thought"].dropna().iloc[0][:200])

**檢查清單再往下:**
- 解析失敗率(上表 `fail_rate`)明顯 > 5%,或輸出 token 常頂到上限 → 先調大 `max_tokens` 或調提示詞,不要放大。
- **時間**:本地模型每人同時 8 個請求。完整實驗每個模型約 1.5 萬次呼叫;依單次回覆秒數,通常是數小時等級。用 5b 的耗時 × 10 估算。
- **費用**:只用免費本地模型 → $0。若啟用付費模型,用試跑花費 × 10 估完整費用(每週額度 $10)。
- 想法內容看起來合理(沒有空白、沒有亂碼)。

### 時間估算
用試跑的實際回覆秒數,估完整實驗要跑多久。**先跑完 5a(或 5b)再執行這格。**
`FULL_REPS` 是完整實驗每格的場數:20 場最完整;想省一半時間改成 10。

In [ ]:
FULL_REPS = 20
import numpy as np
for m in MODELS:
    r = rounds[rounds["model"] == m["name"]]
    s = summary[summary["model"] == m["name"]]
    if r.empty:
        continue
    lat = r["latency_ms"].median() / 1000                  # 單次呼叫的中位數秒數
    nc = r[~r["comm"].eq("chat")] if "comm" in r else r     # 無聊天條件的場次
    calls_per_game = len(nc) / max(nc["game_id"].nunique(), 1)  # 每場約幾次行動呼叫
    n_games_nochat = 20 * 3 * FULL_REPS    # P0–P4(5 條件 × 4 腳本)× 3 賽局
    n_games_chat = 3 * 3 * FULL_REPS       # P5(3 腳本)× 3 賽局,每回合多一次聊天呼叫
    calls = calls_per_game * (n_games_nochat + 2 * n_games_chat)
    conc = m.get("max_concurrent", 2)
    secs = calls * lat / conc
    print(f"{m['name']}: 中位數 {lat:.1f} 秒/次 | 每場約 {calls_per_game:.1f} 次 | 完整約 {calls:,.0f} 次呼叫"
          f" | 同時 {conc} 個 → 約 {secs/3600:.1f} 小時")
print("提醒:這是下限估計(未含重試、尖峰排隊);尖峰時段實際會更慢。")

## 6. 完整實驗(每格 20 場,可重複執行續跑)

In [ ]:
run_all(FULL_REPS, LOGS)

## 7. 分析

In [ ]:
out = subprocess.run([sys.executable, "-m", "coop_llm.analyze_probe", "--logs", LOGS,
                      "--out", "/content/drive/MyDrive/coop-llm-logs/results"], capture_output=True, text=True)
print(out.stdout[-600:], out.stderr[-600:])

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob("/content/drive/MyDrive/coop-llm-logs/results/quality_map_*.png")):
    display(Image(p))

結果檔都在 Drive 的 `coop-llm-logs/results/`(`di_pairs.csv`、`comparisons_*.csv`、圖)。